# Fase 2 - Comprension de los Datos
## Seccion 14: Validacion Cruzada con Datos Oficiales

**Notebook:** notebooks/10_EDA_validacion_oficial.ipynb
**Responsable:** Sofia | **Apoyo:** Steve
**Objetivo:** Validar los precios observados en los datasets del Grupo A contra indices oficiales de precios de vivienda (IPVU/IPVN DANE) para detectar desviaciones sistematicas.

## Configuracion inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
RAW = os.path.join('..', 'data', 'raw')
OUT = 'outputs'
os.makedirs(OUT, exist_ok=True)

---
## Celdas 133-137: Comparar contra IPVU/IPVN DANE

### Cargar IPVN DANE (B1)

In [ ]:
B1 = pd.read_csv(os.path.join(RAW, 'B1_indices_precios_vivienda.csv'), encoding='utf-8-sig')
print('--- B1 - INDICES DE PRECIOS DE VIVIENDA (IPVN) ---')
print(f'Dimensiones: {B1.shape[0]:,} x {B1.shape[1]}')
print(f'Columnas: {list(B1.columns)}')
display(B1.head(10))

### Detectar estructura de IPVN

In [ ]:
print('Analizando columnas de B1...')
year_col = None
index_col = None
city_col = None
for c in B1.columns:
    cl = c.lower().strip()
    if any(x in cl for x in ['ano', 'year', 'periodo', 'trimestre']):
        year_col = c
    elif any(x in cl for x in ['indice', 'ipvu', 'ipvn', 'precio', 'valor']):
        index_col = c
    elif any(x in cl for x in ['ciudad', 'municipio', 'departamento', 'region', 'zona']):
        city_col = c

print(f'Columna temporal: {year_col}')
print(f'Columna indice: {index_col}')
print(f'Columna geografica: {city_col}')

if year_col:
    B1[year_col] = pd.to_numeric(B1[year_col], errors='coerce')
    print(f'\nRango temporal: {B1[year_col].min()} - {B1[year_col].max()}')
if city_col:
    print(f'Ciudades/disponibles: {B1[city_col].nunique()}')

### Convertir IPVN a series anuales

In [ ]:
if year_col and index_col:
    # Si hay columna de ciudad, agrupar por year
    if city_col and city_col in B1.columns:
        annual = B1.groupby([year_col, city_col])[index_col].mean().reset_index()
        annual.columns = ['year', 'ciudad', 'ipvn_medio']
        annual['year'] = annual['year'].astype(int)
        print('IPVN anual por ciudad:')
        display(annual.head(12))
    else:
        annual = B1.groupby(year_col)[index_col].mean().reset_index()
        annual.columns = ['year', 'ipvn_medio']
        annual['year'] = annual['year'].astype(int)
        print('IPVN anual (nacional):')
        display(annual.head(12))
    
    annual.to_csv(os.path.join(OUT, 'ipvn_anual.csv'), index=False)
    print(f'Guardado en outputs/ipvn_anual.csv')
else:
    print('No se pudieron detectar columnas necesarias.')
    annual = None

### Cargar precios de datasets del Grupo A

In [ ]:
A1 = pd.read_csv(os.path.join(RAW, 'A1_colombia_housing_properties.csv'), encoding='utf-8-sig', low_memory=False)
A1['created_on'] = pd.to_datetime(A1['created_on'], errors='coerce')
A1['year'] = A1['created_on'].dt.year
A1['precio'] = A1['price']
A1['ciudad'] = A1['l3'].str.strip().str.title()
print(f'A1: {A1.shape[0]:,} registros')

A2 = pd.read_csv(os.path.join(RAW, 'A2_fincaraiz_colombia.csv'), encoding='utf-8-sig', low_memory=False)
if 'Fecha Actualizacion' in A2.columns:
    A2['Fecha Actualizacion'] = pd.to_datetime(A2['Fecha Actualizacion'], errors='coerce')
    A2['year'] = A2['Fecha Actualizacion'].dt.year
else:
    A2['year'] = None
A2['precio'] = A2['Precio']
A2['ciudad'] = A2['Ciudad'].str.strip().str.title()
print(f'A2: {A2.shape[0]:,} registros')

A5 = pd.read_csv(os.path.join(RAW, 'A5_medellin_properties_2023.csv'), encoding='utf-8-sig', low_memory=False)
A5['year'] = 2023
A5['precio'] = A5['price']
A5['ciudad'] = 'Medellin'
print(f'A5: {A5.shape[0]:,} registros')

### Calcular precio mediano por year desde datasets

In [ ]:
ds_list = [('A1', A1), ('A2', A2), ('A5', A5)]
allds = []
for fid, df in ds_list:
    temp = df[df['precio'] > 0].copy()
    temp = temp[temp['year'].between(2018, 2024)]
    temp['fuente'] = fid
    allds.append(temp[['year', 'precio', 'ciudad', 'fuente']])
df_ds = pd.concat(allds, ignore_index=True)
print(f'Total registros con precio: {len(df_ds):,}')

ds_national = df_ds.groupby('year')['precio'].median().reset_index()
ds_national.columns = ['year', 'precio_mediano_ds']
ds_national['year'] = ds_national['year'].astype(int)
print('\nPrecio mediano datasets por year:')
display(ds_national)

### Comparar precios datasets vs IPVN

In [ ]:
if annual is not None and index_col:
    # Normalizar IPVN a year base
    base_year = 2020
    annual_ipvn = annual.copy()
    
    if 'ciudad' in annual_ipvn.columns:
        # Nacional: promediar ciudades
        ipvn_nac = annual_ipvn.groupby('year')['ipvn_medio'].mean().reset_index()
    else:
        ipvn_nac = annual_ipvn.rename(columns={'ipvn_medio': 'ipvn_medio'})
    
    ipvn_base = ipvn_nac[ipvn_nac['year'] == base_year]['ipvn_medio'].values
    if len(ipvn_base) > 0:
        ipvn_nac['ipvn_normalizado'] = ipvn_nac['ipvn_medio'] / ipvn_base[0] * 100
        
        # Normalizar precio datasets
        ds_base = ds_national[ds_national['year'] == base_year]['precio_mediano_ds'].values
        if len(ds_base) > 0:
            ds_national['precio_normalizado'] = ds_national['precio_mediano_ds'] / ds_base[0] * 100
            
            # Merge
            comp = ipvn_nac.merge(ds_national, on='year', how='inner')
            comp['desviacion_pct'] = ((comp['precio_normalizado'] - comp['ipvn_normalizado']) / comp['ipvn_normalizado']) * 100
            
            print('--- COMPARACION PRECIOS DATASETS vs IPVN OFICIAL ---')
            print(f'Base: {base_year} = 100')
            display(comp.round(1))
            comp.to_csv(os.path.join(OUT, 'comparacion_ipvn_datasets.csv'), index=False)
        else:
            print(f'No hay datos de dataset para year base {base_year}')
            comp = None
    else:
        print(f'No hay datos de IPVN para year base {base_year}')
        comp = None
else:
    print('No se pudo realizar la comparacion.')
    comp = None

### Grafico comparativo: Datasets vs IPVN

In [ ]:
if comp is not None and len(comp) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Panel 1: Indices normalizados
    axes[0].plot(comp['year'], comp['ipvn_normalizado'], marker='s', linewidth=2.5, color='green', label='IPVN DANE (oficial)', markersize=8)
    axes[0].plot(comp['year'], comp['precio_normalizado'], marker='o', linewidth=2.5, color='steelblue', label='Datasets Grupo A', markersize=8)
    axes[0].axhline(100, color='gray', ls='--', alpha=0.5)
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Indice (year base = 100)')
    axes[0].set_title('Comparacion de tendencias: Datasets vs IPVN')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Panel 2: Desviacion
    colors = ['red' if v > 0 else 'green' for v in comp['desviacion_pct']]
    axes[1].bar(comp['year'], comp['desviacion_pct'], color=colors, edgecolor='white', alpha=0.8)
    axes[1].axhline(0, color='black', lw=1)
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Desviacion (%)')
    axes[1].set_title('Desviacion de precios datasets vs IPVN oficial')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, 'comparacion_ipvn_datasets.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Grafico guardado en outputs/comparacion_ipvn_datasets.png')

### Comparacion por ciudad (si IPVN tiene desagregacion)

In [ ]:
if comp is not None and 'ciudad' in annual.columns:
    print('--- COMPARACION POR CIUDAD (ultimo year disponible) ---')
    last_year = annual['year'].max()
    
    # Precio datasets por ciudad
    ds_city = df_ds[df_ds['year'] == last_year].groupby('ciudad')['precio'].median().reset_index()
    ds_city.columns = ['ciudad', 'precio_mediano_ds']
    
    # IPVN por ciudad
    ipvn_city = annual[annual['year'] == last_year][['ciudad', 'ipvn_medio']]
    
    comp_city = ipvn_city.merge(ds_city, on='ciudad', how='inner')
    # Ordenar por precio
    comp_city = comp_city.sort_values('precio_mediano_ds', ascending=False)
    
    print(f'Year: {int(last_year)}')
    display(comp_city)
    
    # Grafico
    plt.figure(figsize=(12, 6))
    x = range(len(comp_city))
    plt.bar(x, comp_city['precio_mediano_ds']/1e6, color='steelblue', alpha=0.7, label='Precio mediano datasets')
    plt.xticks(x, comp_city['ciudad'], rotation=45, ha='right')
    plt.ylabel('Precio (Millones COP)')
    plt.title(f'Precio mediano por ciudad - Datasets ({int(last_year)})')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, 'comparacion_ciudad_ultimo_year.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No hay datos desagregados por ciudad para comparacion.')
    print('(IPVN sin columna geografica o datos insuficientes)')

### Documentar desviaciones y causas

In [ ]:
print('===== DOCUMENTACION DE DESVIACIONES =====\n')

if comp is not None:
    max_dev = comp.loc[comp['desviacion_pct'].abs().idxmax()]
    avg_dev = comp['desviacion_pct'].mean()
    print(f'Desviacion promedio: {avg_dev:+.1f}%')
    print(f'Desviacion maxima:   {max_dev["desviacion_pct"]:+.1f}% en {int(max_dev["year"])}')
    print()

print('Posibles causas de desviacion:')
print()
print('1. Cobertura del IPVN:')
print('   - IPVN DANE cubre solo vivienda nueva (no usada).')
print('   - Datasets incluyen vivienda usada (mayoria).')
print('   - La vivienda usada tiende a precios mas variables.\n')
print('2. Composicion geografica:')
print('   - IPVN tiene cobertura en 8+ ciudades principales.')
print('   - Datasets pueden tener sobrerrepresentacion de ciertas ciudades.\n')
print('3. Segmento de mercado:')
print('   - IPVN captura transacciones reales (escrituras).')
print('   - Datasets capturan precios de oferta (listados), no transaccion.')
print('   - Precio de oferta suele ser 5-15% mayor que precio de transaccion.\n')
print('4. Temporalidad:')
print('   - IPVN es trimestral con rezago de 1-2 trimestres.')
print('   - Datasets tienen fechas de publicacion (no de transaccion).\n')
print('5. Calidad de datos:')
print('   - IPVN usa metodologia estandarizada DANE.')
print('   - Datasets pueden tener errores, duplicados, valores atipicos.')
print('   - La limpieza en Fase 3 reducira estas desviaciones.\n')
print('Conclusion:')
if comp is not None:
    if abs(avg_dev) < 10:
        print('  Los datasets del Grupo A muestran consistencia razonable con IPVN')
        print(f'  (desviacion media de {avg_dev:+.1f}%).')
        print('  Las tendencias de precio son validas para el analisis exploratorio.')
    else:
        print(f'  Los datasets muestran desviacion significativa ({avg_dev:+.1f}%).')
        print('  Se recomienda ajuste o factor de correccion en Fase 3.')

---
## Resumen de la Seccion 14

- Se cargo IPVN DANE (B1) y se convirtio a series anuales.
- Se calcularon precios medianos de datasets del Grupo A (A1, A2, A5).
- Se normalizaron ambas series a year base 2020 para comparacion.
- Se grafico tendencia comparativa y desviacion porcentual.
- Se documento causas de desviacion (cobertura, oferta vs transaccion, segmento).

**Outputs generados:**
- outputs/ipvn_anual.csv
- outputs/comparacion_ipvn_datasets.csv
- outputs/comparacion_ipvn_datasets.png
- outputs/comparacion_ciudad_ultimo_year.png